In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv('titanic_full.csv')[['Age','Pclass','Fare','Survived']]

In [4]:
df.head()

,Age,Pclass,Fare,Survived
0,22.0,3,7.2500,0
1,38.0,1,71.2833,1
2,26.0,3,7.9250,1
3,35.0,1,53.1000,1
4,35.0,3,8.0500,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       714 non-null    float64
 1   Pclass    891 non-null    int64  
 2   Fare      891 non-null    float64
 3   Survived  891 non-null    int64  
dtypes: float64(2), int64(2)
memory usage: 28.0 KB


In [7]:
print(df.isnull().mean() * 100)

Age         19.86532
Pclass       0.00000
Fare         0.00000
Survived     0.00000
dtype: float64


- We see that we have Null values inside Age Feature Only
- Firstly we will split the dataframe into Feature dataset and Target Series

In [8]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [9]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2) # Train test split to prevent Data Leakage

In [10]:
X_train.head()

,Age,Pclass,Fare
30,40.0,1,27.7208
10,4.0,3,16.7000
873,47.0,3,9.0000
182,9.0,3,31.3875
876,20.0,3,9.8458


### ***Import KNNImputer Class***

In [11]:
knn = KNNImputer(n_neighbors=3,weights='distance') 
# n_neighbours = 3 means we will look for 3 nearest rows.
# weights = 'distance' means the nearer row has more influence (1/distance formula) 

X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)

In [12]:
X_train_trf 

array([[ 40.        ,   1.        ,  27.7208    ],
       [  4.        ,   3.        ,  16.7       ],
       [ 47.        ,   3.        ,   9.        ],
       ...,
       [ 71.        ,   1.        ,  49.5042    ],
       [ 32.66666667,   1.        , 221.7792    ],
       [ 49.76289518,   1.        ,  25.925     ]], shape=(712, 3))

- See that after transforming -> it became numpy array
- To get the output directly into dataframe format, we use -> 

In [ ]:
knn = KNNImputer(n_neighbors=1,weights='distance' , add_indicator=True) 
# n_neighbours = 3 means we will look for 3 nearest rows.
# weights = 'distance' means the nearer row has more influence (1/distance formula) 

# add_indicator=True appends a binary flag column that marks missing values:
# 0 -> the original value was present
# 1 -> the original value was missing and had to be imputed

knn.set_output(transform='pandas')

X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)


In [38]:
X_train_trf

,Age,Pclass,Fare,missingindicator_Age
30,40.0,1.0,27.7208,0.0
10,4.0,3.0,16.7000,0.0
873,47.0,3.0,9.0000,0.0
182,9.0,3.0,31.3875,0.0
876,20.0,3.0,9.8458,0.0
...,...,...,...,...
534,30.0,3.0,8.6625,0.0
584,31.0,3.0,8.7125,1.0
493,71.0,1.0,49.5042,0.0
527,18.0,1.0,221.7792,1.0


### ***Model Evaluation Using KNN Imputer***

In [39]:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred)

0.7206703910614525

### ***Model Evaluation Using Simple Mean Imputer***

In [40]:
# Comparison with Simple Imputer --> mean

si = SimpleImputer(add_indicator=True)

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [45]:
lr = LogisticRegression()

lr.fit(X_train_trf2,y_train)

y_pred2 = lr.predict(X_test_trf2)

accuracy_score(y_test,y_pred2)

0.6983240223463687

- **We see a small improvement by just changing the method of Imputation from Mean -> KNN**